# FMA Spectrogram Preprocessing & Data Cleaning

This notebook cleans the FMA metadata, emits per-subset CSVs, and turns the `fma_small` MP3s into log-mel spectrograms saved to disk.

Two cleaned subsets are written:
- **small** — `subset == "small"` (≈8 000 tracks). Used for spectrograms and any small-only model. Drops missing/undecodable audio + the `FAILED` track ids from `creation.ipynb`.
- **medium** — `subset ∈ {small, medium}` (≈25 000 tracks; FMA medium is a superset of small). RF-friendly: drops null genre, the `FAILED` list, and the few corrupt small MP3s we already know about. No per-track audio decode check (we don't have `fma_medium/` on disk).

Spectrograms are only extracted for the small subset. Random Forest can train on either subset by picking the right CSV.

## 1. Setup

Run once if any of these are missing:

```python
%pip install torch torchaudio pandas numpy tqdm soundfile
```

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# Resolve paths from either the project root or the code/ subdirectory.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_CANDIDATES:
    audio_dir = candidate / "fma_small"
    metadata_path = candidate / "fma_metadata" / "tracks.csv"
    if audio_dir.exists() and metadata_path.exists():
        PROJECT_DIR = candidate.resolve()
        AUDIO_DIR = audio_dir.resolve()
        METADATA_PATH = metadata_path.resolve()
        break
else:
    raise FileNotFoundError("Could not find fma_small/ and fma_metadata/tracks.csv.")

# Where the cleaned CSVs and spectrograms get written.
PROCESSED_DIR = PROJECT_DIR / "fma_preprocessed"
SPECTROGRAM_DIR = PROCESSED_DIR / "spectrograms"
SPECTROGRAM_DIR.mkdir(parents=True, exist_ok=True)

# Audio validation cache produced by earlier notebooks; reuse it if present.
AUDIO_VALIDATION_CACHE = PROJECT_DIR / "fma_small_audio_validation.csv"

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print({
    "PROJECT_DIR": str(PROJECT_DIR),
    "AUDIO_DIR": str(AUDIO_DIR),
    "METADATA_PATH": str(METADATA_PATH),
    "PROCESSED_DIR": str(PROCESSED_DIR),
    "SPECTROGRAM_DIR": str(SPECTROGRAM_DIR),
    "device": device,
})

{'PROJECT_DIR': '/home/lily/acml-project', 'AUDIO_DIR': '/home/lily/acml-project/fma_small', 'METADATA_PATH': '/home/lily/acml-project/fma_metadata/tracks.csv', 'PROCESSED_DIR': '/home/lily/acml-project/fma_preprocessed', 'SPECTROGRAM_DIR': '/home/lily/acml-project/fma_preprocessed/spectrograms', 'device': 'cpu'}


/home/lily/acml-project/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Load `tracks.csv`

`tracks.csv` uses a two-row header, so columns come back as a `MultiIndex` like `("track", "genre_top")`. We pull only the fields we need and flatten them into a normal DataFrame.

In [2]:
def track_id_to_audio_path(track_id: int) -> Path:
    track_id = int(track_id)
    filename = f"{track_id:06d}.mp3"
    return AUDIO_DIR / filename[:3] / filename


tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

# metadata_all keeps every track so we can derive both the small and medium cleaned subsets below.
metadata_all = pd.DataFrame({
    "track_id": tracks.index.astype(int),
    "subset": tracks[("set", "subset")],
    "split": tracks[("set", "split")],
    "genre": tracks[("track", "genre_top")],
    "duration": tracks[("track", "duration")],
    "bit_rate": tracks[("track", "bit_rate")],
    "title": tracks[("track", "title")],
    "artist": tracks[("artist", "name")],
})
metadata_all["audio_path"] = metadata_all["track_id"].apply(track_id_to_audio_path)

print(f"Total tracks in tracks.csv: {len(metadata_all):,}")
print(metadata_all["subset"].value_counts())
metadata_all.head()

Total tracks in tracks.csv: 106,574
subset
large     81574
medium    17000
small      8000
Name: count, dtype: int64


,track_id,subset,split,genre,duration,bit_rate,title,artist,audio_path
track_id,,,,,,,,,
2,2,small,training,Hip-Hop,168,256000,Food,AWOL,/home/lily/acml-project/fma_small/000/000002.mp3
3,3,medium,training,Hip-Hop,237,256000,Electric Ave,AWOL,/home/lily/acml-project/fma_small/000/000003.mp3
5,5,small,training,Hip-Hop,206,256000,This World,AWOL,/home/lily/acml-project/fma_small/000/000005.mp3
10,10,small,training,Pop,161,192000,Freeway,Kurt Vile,/home/lily/acml-project/fma_small/000/000010.mp3
20,20,large,training,NaN,311,256000,Spiritual Level,Nicky Cook,/home/lily/acml-project/fma_small/000/000020.mp3


## 3. Clean each subset

`creation.ipynb` does its cleanup with a small `keep(...)` helper that prints how many rows survive each step. We reuse that pattern so it is obvious what each filter drops.

We build two cleaned frames:
- `metadata` — the **small** subset (with full audio validation), used downstream for spectrogram extraction.
- `metadata_medium` — the **medium** subset (small ∪ medium-only), CSV-only.

In [3]:
# Same `keep` helper used in creation.ipynb.
def keep(mask_or_index, df, label):
    old = len(df)
    df = df.loc[mask_or_index] if mask_or_index.dtype != bool else df[mask_or_index]
    print(f"{label:35s} {old - len(df):>5d} dropped, {len(df):>5d} left")
    return df


print("=== Cleaning small ===")
metadata = metadata_all.copy()

# 1. small subset only
metadata = keep(metadata["subset"] == "small", metadata, "subset == small")

# 2. must have a top-level genre
metadata = keep(metadata["genre"].notna(), metadata, "genre_top not null")

# 3. audio file must exist on disk
exists_mask = metadata["audio_path"].map(lambda p: p.exists())
metadata = keep(exists_mask, metadata, "audio file present on disk")

=== Cleaning small ===
subset == small                     98574 dropped,  8000 left
genre_top not null                      0 dropped,  8000 left
audio file present on disk              0 dropped,  8000 left


In [4]:
def can_decode_audio(path: Path) -> tuple[bool, str]:
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        if sample_rate <= 0 or audio.size == 0:
            return False, "empty_audio"
        return True, ""
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def build_audio_validation(paths: pd.Series) -> pd.DataFrame:
    """Return a DataFrame indexed by audio_path with is_valid/error columns.

    Uses the cached CSV when it covers every path; otherwise rebuilds it.
    """
    cached = None
    if AUDIO_VALIDATION_CACHE.exists():
        try:
            cached = pd.read_csv(AUDIO_VALIDATION_CACHE)
            if {"audio_path", "is_valid", "error"}.issubset(cached.columns):
                cached = cached.drop_duplicates(subset=["audio_path"], keep="last")
            else:
                cached = None
        except Exception:
            cached = None

    paths_str = set(paths.map(str))
    if cached is not None and paths_str.issubset(set(cached["audio_path"])):
        return cached.set_index("audio_path")

    rows = []
    for path in tqdm(paths, desc="Validating audio"):
        is_valid, error = can_decode_audio(path)
        rows.append({"audio_path": str(path), "is_valid": is_valid, "error": error})
    validation = pd.DataFrame(rows).drop_duplicates(subset=["audio_path"], keep="last")
    validation.to_csv(AUDIO_VALIDATION_CACHE, index=False)
    return validation.set_index("audio_path")


validation = build_audio_validation(metadata["audio_path"])

metadata["audio_path_str"] = metadata["audio_path"].map(str)
metadata = metadata.join(validation, on="audio_path_str")

bad = metadata[~metadata["is_valid"].fillna(False)]
if len(bad):
    print(f"\n{len(bad)} undecodable files:")
    print(bad[["track_id", "split", "genre", "audio_path", "error"]].to_string(index=False))

metadata = keep(metadata["is_valid"].fillna(False), metadata, "audio decodable")
metadata = metadata.drop(columns=["audio_path_str", "is_valid", "error"])

Validating audio:   0%|          | 0/8000 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


KeyboardInterrupt: 

In [ ]:
# Track ids that creation.ipynb documents as feature-extraction failures
# (ffmpeg header issues, librosa "filter pass-band lies beyond Nyquist", etc.).
FAILED = [
    1440, 26436, 28106, 29166, 29167, 29168, 29169, 29170, 29171, 29172,
    29173, 29179, 38903, 43903, 56757, 57603, 59361, 62095, 62954, 62956,
    62957, 62959, 62971, 75461, 80015, 86079, 92345, 92346, 92347, 92348,
    92349, 92350, 92351, 92352, 92353, 92354, 92355, 92356, 92357, 92358,
    92359, 92360, 92361, 96426, 104623, 106719, 109714, 114448, 114501, 114528,
    115235, 117759, 118003, 118004, 127827, 130296, 130298, 131076, 135804, 136486,
    144769, 144770, 144771, 144773, 144774, 144775, 144776, 144777, 144778, 152204,
    154923,
]

metadata = keep(~metadata["track_id"].isin(FAILED), metadata, "not in FAILED list")

# Final tidy: stable ordering, integer label per genre, reset index.
metadata = metadata.sort_values("track_id").reset_index(drop=True)
genres_small_sorted = sorted(metadata["genre"].unique())
genre_to_idx_small = {g: i for i, g in enumerate(genres_small_sorted)}
metadata["label"] = metadata["genre"].map(genre_to_idx_small).astype(int)

print("\nSmall genre distribution:")
print(metadata["genre"].value_counts().sort_index())
print("\nSmall split distribution:")
print(metadata["split"].value_counts())

In [ ]:
print("=== Cleaning medium ===")
metadata_medium = metadata_all.copy()

# FMA convention: medium ⊃ small. Use isin so the cleaned medium frame has all 25k tracks
# (8000 small + 17000 medium-only), matching the FMA paper's medium definition.
metadata_medium = keep(
    metadata_medium["subset"].isin(["small", "medium"]),
    metadata_medium,
    "subset in {small, medium}",
)
metadata_medium = keep(metadata_medium["genre"].notna(), metadata_medium, "genre_top not null")
metadata_medium = keep(~metadata_medium["track_id"].isin(FAILED), metadata_medium, "not in FAILED list")

# We can't decode-check the medium-only tracks (we don't have fma_medium/), but we already know
# which small-subset MP3s are corrupt. Drop those ids so the medium frame is consistent with the
# small frame on the shared portion.
small_corrupt_ids = sorted(set(metadata_all["track_id"]).difference(metadata["track_id"]))
small_corrupt_ids = [tid for tid in small_corrupt_ids
                     if tid in set(metadata_medium["track_id"]) and tid not in FAILED]
if small_corrupt_ids:
    metadata_medium = keep(
        ~metadata_medium["track_id"].isin(small_corrupt_ids),
        metadata_medium,
        "small-subset audio decodable",
    )

metadata_medium = metadata_medium.sort_values("track_id").reset_index(drop=True)
genres_medium_sorted = sorted(metadata_medium["genre"].unique())
genre_to_idx_medium = {g: i for i, g in enumerate(genres_medium_sorted)}
metadata_medium["label"] = metadata_medium["genre"].map(genre_to_idx_medium).astype(int)

print("\nMedium genre distribution:")
print(metadata_medium["genre"].value_counts().sort_index())
print("\nMedium split distribution:")
print(metadata_medium["split"].value_counts())

## 4. Save the cleaned CSVs

For each subset we write:
- `tracks_clean_{subset}.csv` — the full cleaned frame.
- `tracks_clean_{subset}_{split}.csv` — per-split slices so training notebooks can read just what they need.
- `genre_to_idx_{subset}.csv` — the genre → label mapping.

In [ ]:
def save_clean_csvs(frame: pd.DataFrame, subset_name: str, genre_to_idx: dict[str, int]) -> None:
    base_path = PROCESSED_DIR / f"tracks_clean_{subset_name}.csv"
    frame.to_csv(base_path, index=False)
    print(f"Wrote {base_path} ({len(frame):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = frame[frame["split"] == split_name]
        split_path = PROCESSED_DIR / f"tracks_clean_{subset_name}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    genre_map_path = PROCESSED_DIR / f"genre_to_idx_{subset_name}.csv"
    pd.DataFrame(
        sorted(genre_to_idx.items(), key=lambda kv: kv[1]),
        columns=["genre", "label"],
    ).to_csv(genre_map_path, index=False)
    print(f"Wrote {genre_map_path}")


save_clean_csvs(metadata, "small", genre_to_idx_small)
print()
save_clean_csvs(metadata_medium, "medium", genre_to_idx_medium)

## 5. Spectrogram settings

Same numbers as `audio_encoder.ipynb` / `cnn_vs_transformer.ipynb` so the saved spectrograms are drop-in compatible:

- 16 kHz mono
- 10 s clip (centered) → 160 000 samples
- STFT: `n_fft=400`, `hop=160` (≈ 25 ms window, 10 ms hop)
- 64 mel bins, log power scale, per-clip mean/std normalization
- Output shape: `(1, 64, 1001)` saved as `float32`

In [ ]:
SAMPLE_RATE = 16_000
CLIP_SECONDS = 10
NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS
N_FFT = 400
HOP_LENGTH = 160
N_MELS = 64

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(device)


def load_clip(path: Path) -> torch.Tensor:
    """Read an MP3, convert to mono 16 kHz, center-crop or pad to NUM_SAMPLES."""
    audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)

    if sample_rate != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

    if waveform.shape[1] > NUM_SAMPLES:
        start = (waveform.shape[1] - NUM_SAMPLES) // 2
        waveform = waveform[:, start:start + NUM_SAMPLES]
    elif waveform.shape[1] < NUM_SAMPLES:
        waveform = torch.nn.functional.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

    waveform = torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)
    return waveform.clamp(-1.0, 1.0)


@torch.no_grad()
def waveform_to_spec(waveform: torch.Tensor) -> torch.Tensor:
    waveform = waveform.to(device)
    spec = to_db(mel_transform(waveform))
    spec = torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)
    mean = spec.mean(dim=(-2, -1), keepdim=True)
    std = spec.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
    spec = (spec - mean) / std
    return torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0).cpu()


# Sanity check on one clip.
sample_row = metadata.iloc[0]
sample_wave = load_clip(sample_row["audio_path"])
sample_spec = waveform_to_spec(sample_wave)
print(f"waveform shape: {tuple(sample_wave.shape)}")
print(f"spectrogram shape: {tuple(sample_spec.shape)}")
print(f"sample track: {sample_row['track_id']} ({sample_row['genre']})")

## 6. Extract and save every spectrogram

Output layout mirrors `fma_small/`:

```
fma_preprocessed/
  spectrograms/
    000/000002.npy
    000/000005.npy
    ...
```

Already-extracted spectrograms are skipped so the cell is safe to re-run. Tracks that throw during decode/STFT are recorded in a final `dropped` list and removed from the manifest.

In [ ]:
def track_id_to_spec_path(track_id: int) -> Path:
    filename = f"{int(track_id):06d}.npy"
    return SPECTROGRAM_DIR / filename[:3] / filename


# Toggle to False to recompute everything from scratch.
SKIP_EXISTING = True

spec_paths = []
dropped_rows = []
n_skipped = 0
n_written = 0

for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Extracting"):
    out_path = track_id_to_spec_path(row.track_id)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if SKIP_EXISTING and out_path.exists():
        spec_paths.append(str(out_path))
        n_skipped += 1
        continue

    try:
        waveform = load_clip(Path(row.audio_path))
        spec = waveform_to_spec(waveform).numpy().astype(np.float32)
    except Exception as exc:
        dropped_rows.append({
            "track_id": row.track_id,
            "audio_path": str(row.audio_path),
            "error": f"{type(exc).__name__}: {exc}",
        })
        spec_paths.append(None)
        continue

    np.save(out_path, spec)
    spec_paths.append(str(out_path))
    n_written += 1

print(f"\nWrote {n_written:,} new spectrograms, reused {n_skipped:,} existing, "
      f"dropped {len(dropped_rows):,} on error.")
if dropped_rows:
    print(pd.DataFrame(dropped_rows).head(10).to_string(index=False))

## 7. Save the spectrogram manifest

`spectrograms_manifest.csv` is the canonical pointer a training notebook should consume: one row per usable clip with the `.npy` path, label, split, and original audio path.

In [ ]:
manifest = metadata.copy()
manifest["spectrogram_path"] = spec_paths
manifest = manifest[manifest["spectrogram_path"].notna()].reset_index(drop=True)

manifest_columns = [
    "track_id", "split", "genre", "label",
    "spectrogram_path", "audio_path",
    "duration", "bit_rate", "title", "artist",
]
manifest = manifest[manifest_columns]

manifest_path = PROCESSED_DIR / "spectrograms_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path} ({len(manifest):,} rows)")

for split_name in ("training", "validation", "test"):
    split_manifest = manifest[manifest["split"] == split_name]
    split_path = PROCESSED_DIR / f"spectrograms_manifest_{split_name}.csv"
    split_manifest.to_csv(split_path, index=False)
    print(f"Wrote {split_path} ({len(split_manifest):,} rows)")

if dropped_rows:
    dropped_path = PROCESSED_DIR / "spectrogram_extraction_errors.csv"
    pd.DataFrame(dropped_rows).to_csv(dropped_path, index=False)
    print(f"Wrote {dropped_path} ({len(dropped_rows):,} rows)")

## 8. Quick sanity check

Load one `.npy` back and confirm the shape/stats match the in-memory tensor we generated above.

In [ ]:
first = manifest.iloc[0]
loaded = np.load(first["spectrogram_path"])
print(f"track_id: {first['track_id']} | genre: {first['genre']} | split: {first['split']}")
print(f"shape: {loaded.shape} | dtype: {loaded.dtype}")
print(f"mean: {loaded.mean():.4f} | std: {loaded.std():.4f} | "
      f"min: {loaded.min():.2f} | max: {loaded.max():.2f}")